## 延后初始化
在前面的学习中。我们没有考虑这些问题
* 定义了网络框架，但没有指定输入维度
* 添加层时没有指定前一层的输入维度
* 进行初始化参数的时候没有足够的信息来确认
> 数据第一次经过网络时，框架才会动态推断处每个层的大小。所以我们需要延后进行数据初始化
> LazyLinear->Linear的拖延症版本  
特性,nn.Linear (普通),nn.LazyLinear (懒惰)
初始化时机,定义时立即创建参数,第一次执行前向传播时创建
必需参数,in_features 和 out_features,仅需 out_features
容错性,算错一个数程序就报错,自动适配，不容易出错
内存占用,定义后立即占用,数据进来前几乎不占内存

In [5]:
from torch import nn
import torch
from d2l import torch as d2l
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))#lazylinear的输入维度未知，无法初始化权重和偏置参数,只需传入输出维度

In [6]:
X = torch.rand(2,20)
net(X)#现在LazyLinear被初始化了，输出不再是None
net[0].weight, net[0].weight.shape#这里可以看到第一层的权重参数已经被初始化了，输入维度是20，输出维度是256，注意是反过来的

(Parameter containing:
 tensor([[-0.0235, -0.2038, -0.0168,  ..., -0.1226,  0.1006, -0.1199],
         [-0.1133,  0.1115, -0.0260,  ..., -0.0726,  0.0602, -0.1050],
         [ 0.0511,  0.0547,  0.0777,  ...,  0.0704,  0.0279, -0.0095],
         ...,
         [-0.1955,  0.1486, -0.1880,  ..., -0.0443, -0.1806, -0.0066],
         [ 0.2211,  0.0615, -0.1683,  ...,  0.0507,  0.0683, -0.1356],
         [ 0.0415, -0.0298, -0.1713,  ...,  0.0967, -0.0239,  0.1875]],
        requires_grad=True),
 torch.Size([256, 20]))

In [ ]:
def apply_init(self,inputs,init=None):
    self.forward(*inputs)
    if init is not None:
        self.net.apply(init)

| 代码片段 | 语法角色 | 实际传入的内容 | 通俗比喻 |
|---|---|---|---|
| `self` | 实例引用 | 模型对象本身 | “我”自己 |
| `inputs` | 位置参数 | 一个包含数据的元组（如 `(X,)`） | 一箱待处理的零件 |
| `*inputs` | 参数解包 | 元组里的具体张量数据 `X` | 把零件从箱子里拿出来用 |
| `init` | 关键字参数 | 某种初始化算法（函数对象） | 一桶指定颜色的油漆 |

## 代码工作流程（重点：延后初始化 Lazy Initialization）

### 1. 定义网络时：只确定“输出维度”，不立即创建参数
```python
net = nn.Sequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
)
```

此时两层 `LazyLinear` 还不知道输入特征数（`in_features`），所以：
- 权重和偏置参数**还没有真正分配**
- 框架只记录“未来要有一层输出 256 / 10 的线性层”

这就是“延后初始化”的核心：**先搭结构，参数等看到真实数据再定。**

---

### 2. 准备输入数据
```python
X = torch.rand(2, 20)
```
`X` 的形状是 `(batch_size=2, feature_dim=20)`。  
这里的 `20` 会成为第一层线性层的输入维度依据。

---

### 3. 第一次前向传播：触发真正初始化
```python
net(X)
```
***前向传播一次之后，自动完成初始化***
第一次执行时发生两件事：

1. **第1个 LazyLinear(256)**  
   - 从 `X.shape[1]` 推断 `in_features=20`
   - 立即转成普通 `Linear(20, 256)` 并创建参数  
   - 权重形状为 `(256, 20)`，偏置形状为 `(256,)`

2. **第2个 LazyLinear(10)**  
   - 它接收上一层输出（特征维 256）
   - 推断 `in_features=256`
   - 转成 `Linear(256, 10)` 并创建参数  
   - 权重形状为 `(10, 256)`，偏置形状为 `(10,)`

所以你现在看到的 `net` 已经是：
- `Linear(20,256) -> ReLU -> Linear(256,10)`

---

### 4. 为什么这样做有价值
- 减少手动计算输入维度的负担
- 降低“维度写错导致报错”的概率
- 在模型刚定义时几乎不占参数内存（参数延后到首个 forward 才创建）
- 对快速搭建原型非常友好

---

### 5. `apply_init` 这段函数在流程中的意义
自动跑完前向传播，拿到数据，自动lazylinear自动完成初始化，转化成正常的linear函数   
如果用户传入了init参数，就按照用户的来
```python
def apply_init(self, inputs, init=None):
    self.forward(*inputs)
    if init is not None:
        self.net.apply(init)
```

逻辑是：
，确保 `LazyLinear` 先变成真正的 `Linear`（参数张量已创建、形状已确定）。  

`self.net.apply(init)` 的意义是：把你传入的 `init` 函数**递归应用到 `net` 的每个子层**。  
它的调用方式等价于：对每个模块执行一次 `init(module)`。

可理解为这 3 步：

1. `self.forward(*inputs)`：只做一件事——让懒初始化层根据输入形状创建参数（如 `weight`/`bias`）。
2. `self.net.apply(init)`：遍历各层，把 `init` 函数作用到已有参数上。
3. `init` 通常用 `nn.init.*` 对参数做**原地改写**（in-place），例如 `xavier_uniform_`、`kaiming_uniform_`。

关键点：  
- 输入 `X` 只用于**推断参数形状**，不会“变成参数再写回去”。  
- 真正“怎么初始化”由 `init` 函数决定；`apply` 只是批量调用它。  

例如常见写法（供说明）：
- 若层是 `nn.Linear`，则对 `m.weight` 做 `xavier_uniform_`，对 `m.bias` 置零。
2. 再执行 `init`，此时参数已经存在，初始化函数才能真正作用到权重上

即：**先“让参数出现”，再“对参数做初始化”。**
### 5.1 关键澄清：`self.forward(*inputs)` **不会**让 `init` 从 `None` 变成非空

`init` 是否为空，只取决于**调用函数时有没有传入初始化函数**，与 `forward` 本身无关。  

可以这样理解：

- `self.forward(*inputs)` 的作用：让 `LazyLinear` 完成参数创建（把“占位层”变成真正有权重的层）
- `if init is not None:` 的作用：检查“调用者是否提供了初始化策略”
- `self.net.apply(init)` 的作用：当且仅当提供了 `init` 时，把该初始化函数应用到已创建好的参数上

所以正确因果是：

1. 先 `forward`，确保参数存在  
2. 再看 `init` 是否非空（这是外部传参决定的）  
3. 若非空，才能安全初始化参数

> 结论：`forward` 改变的是“网络参数状态”，不是 `init` 变量本身。
---